# 🧮 Odometer Pilot: LLM Counting Failure Study — Qwen2.5

Appendix replication of the Llama-3.2 odometer study on the Qwen2.5 family.

| Model | Change MODEL_NAME to | n_layers |
|-------|---------------------|----------|
| Qwen2.5-1.5B-Instruct | `Qwen/Qwen2.5-1.5B-Instruct` | 28 |
| Qwen2.5-3B-Instruct | `Qwen/Qwen2.5-3B-Instruct` | 36 |
| Qwen2.5-7B-Instruct | `Qwen/Qwen2.5-7B-Instruct` | 28 |

**Only cell 1 (config) needs to change between model runs.**
All other cells are model-agnostic within the Qwen2.5 family.

## 0 · Install dependencies

In [1]:
%pip install transformers torch accelerate nbformat matplotlib scikit-learn --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
from huggingface_hub import login
login()

## 1 · Config

**THIS IS THE ONLY CELL TO CHANGE BETWEEN MODEL RUNS.**

```
Qwen2.5-1.5B-Instruct  ->  MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
Qwen2.5-3B-Instruct    ->  MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
Qwen2.5-7B-Instruct    ->  MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
```

CRITICAL_LAYERS is set as a placeholder — update it after running the logit lens
(cell 14) to the 5 layers surrounding the lock-in point for this model.

In [20]:
import re
import json
import random
from collections import Counter
from typing import Optional

import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# ── CHANGE THIS PER RUN ───────────────────────────────────────────────────
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
# MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
# MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# Set after running logit lens (cell 14) — update to layers around lock-in point
# Placeholder values shown; replace with actual lock-in layer ± 2
# Example: if logit lens shows lock-in at L22, set [20, 21, 22, 23, 24]

# CRITICAL_LAYERS = None  # set after logit lens
CRITICAL_LAYERS = [20, 21, 22, 23, 24, 25, 26]

# ── Fixed config — do not change ─────────────────────────────────────────
N_RUNS         = 10
SEEDS          = list(range(N_RUNS))
TEMPERATURE    = 0.0
MAX_NEW_TOKENS = 16
DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"

# Short model tag for output filenames
MODEL_TAG = MODEL_NAME.split("/")[-1].lower().replace("-", "_")

print(f"Model    : {MODEL_NAME}")
print(f"Tag      : {MODEL_TAG}")
print(f"Device   : {DEVICE}")
print(f"N_RUNS   : {N_RUNS}, TEMPERATURE: {TEMPERATURE}")

Model    : Qwen/Qwen2.5-1.5B-Instruct
Tag      : qwen2.5_1.5b_instruct
Device   : cuda
N_RUNS   : 10, TEMPERATURE: 0.0


## 2 · Define prompts

In [4]:
PROMPTS = {
    "phase1_baseline": {
        "text": (
            'Count the number of times "apple" appears in this list: '
            "apple apple apple apple apple apple apple apple apple apple. "
            "Respond only with the integer, nothing else."
        ),
        "expected": 10,
        "description": "10 identical tokens (apple x10)",
    },
    "phase2_anomaly": {
        "text": (
            'Count the number of times "apple" appears in this list: '
            "apple apple apple apple banana apple apple apple apple apple. "
            "Respond only with the integer, nothing else."
        ),
        "expected": 9,
        "description": "9 apple + 1 banana at position 5",
    },
    "phase3_control": {
        "text": (
            "Count the number of words in this list: "
            "dog cat car red blue green house tree book pen. "
            "Respond only with the integer, nothing else."
        ),
        "expected": 10,
        "description": "10 unique tokens (distinct words)",
    },
}

# Fixed prompts — comma-separated, no 'comma-separated' in instruction wording
PROMPTS_FIXED = {
    "phase1_baseline": {
        "text": (
            'Count the number of times "apple" appears in this list: '
            "apple, apple, apple, apple, apple, apple, apple, apple, apple, apple. "
            "Respond only with the integer, nothing else."
        ),
        "expected": 10,
        "description": "10 identical tokens, comma-separated",
    },
    "phase2_anomaly": {
        "text": (
            'Count the number of times "apple" appears in this list: '
            "apple, apple, apple, apple, banana, apple, apple, apple, apple, apple. "
            "Respond only with the integer, nothing else."
        ),
        "expected": 9,
        "description": "9 apple + 1 banana at position 5, comma-separated",
    },
    "phase3_control": {
        "text": (
            "Count the number of words in this list: "
            "dog, cat, car, red, blue, green, house, tree, book, pen. "
            "Respond only with the integer, nothing else."
        ),
        "expected": 10,
        "description": "10 unique tokens, comma-separated",
    },
}

for k, v in PROMPTS.items():
    print(f"[{k}]\n  Expected : {v['expected']}\n  Prompt   : {v['text'][:80]}...\n")

[phase1_baseline]
  Expected : 10
  Prompt   : Count the number of times "apple" appears in this list: apple apple apple apple ...

[phase2_anomaly]
  Expected : 9
  Prompt   : Count the number of times "apple" appears in this list: apple apple apple apple ...

[phase3_control]
  Expected : 10
  Prompt   : Count the number of words in this list: dog cat car red blue green house tree bo...



## 3 · Load model & tokenizer

In [5]:
print(f"Loading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
)

n_layers = model.config.num_hidden_layers
print(f"Model loaded. Parameters: {sum(p.numel() for p in model.parameters())/1e9:.2f}B")
print(f"Layers: {n_layers}")
print(f"Hidden dim: {model.config.hidden_size}")

Loading Qwen/Qwen2.5-1.5B-Instruct...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded. Parameters: 1.54B
Layers: 28
Hidden dim: 1536


## 4 · Helpers

In [6]:
def extract_count(raw_output: str) -> Optional[int]:
    match = re.search(r'\b(\d+)\b', raw_output.strip())
    return int(match.group(1)) if match else None


def get_top_digit(logits_1d):
    """Find digit token (1-20) with highest logit. Handles single-token digits only."""
    candidates = {}
    for n in range(1, 21):
        ids = tokenizer.encode(str(n), add_special_tokens=False)
        if len(ids) == 1:
            candidates[str(n)] = logits_1d[ids[0]].item()
    return max(candidates, key=candidates.get) if candidates else "?"


def make_inputs(prompt_text):
    """Apply chat template and return input dict on model device."""
    messages = [{"role": "user", "content": prompt_text}]
    return tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)


def remove_all_hooks(m):
    for module in m.modules():
        module._forward_hooks.clear()
        module._forward_pre_hooks.clear()
        module._backward_hooks.clear()


# Sanity checks
assert extract_count("10") == 10
assert extract_count("The answer is 9.") == 9
assert extract_count("no number") is None

# Verify digit tokenization for this model
print("Digit token check (single-token digits only):")
for n in range(1, 16):
    ids = tokenizer.encode(str(n), add_special_tokens=False)
    tag = "OK" if len(ids) == 1 else f"MULTI-TOKEN ({ids})"
    print(f"  {n:>3} -> {ids} -> {tag}")

print("\nHelpers OK.")

Digit token check (single-token digits only):
    1 -> [16] -> OK
    2 -> [17] -> OK
    3 -> [18] -> OK
    4 -> [19] -> OK
    5 -> [20] -> OK
    6 -> [21] -> OK
    7 -> [22] -> OK
    8 -> [23] -> OK
    9 -> [24] -> OK
   10 -> [16, 15] -> MULTI-TOKEN ([16, 15])
   11 -> [16, 16] -> MULTI-TOKEN ([16, 16])
   12 -> [16, 17] -> MULTI-TOKEN ([16, 17])
   13 -> [16, 18] -> MULTI-TOKEN ([16, 18])
   14 -> [16, 19] -> MULTI-TOKEN ([16, 19])
   15 -> [16, 20] -> MULTI-TOKEN ([16, 20])

Helpers OK.


## 5 · Run experiment — original prompts

In [7]:
def run_single(phase_key, seed):
    entry  = PROMPTS[phase_key]
    prompt = entry["text"]

    torch.manual_seed(seed)
    random.seed(seed)
    np.random.seed(seed)

    raw = pipe(
        [{"role": "user", "content": prompt}],
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
        do_sample=False,
        pad_token_id=pipe.tokenizer.eos_token_id,
        return_full_text=False,
    )[0]["generated_text"].strip()

    predicted = extract_count(raw)
    correct   = (predicted == entry["expected"]) if predicted is not None else False
    return {"seed": seed, "raw": raw, "predicted": predicted,
            "expected": entry["expected"], "correct": correct}


all_results = {}

for phase_key, entry in PROMPTS.items():
    print(f"\n{chr(9472)*55}")
    print(f"  {phase_key.upper()}  |  {entry['description']}")
    print(f"{chr(9472)*55}")
    phase_results = []
    for seed in SEEDS:
        r      = run_single(phase_key, seed)
        status = "✓" if r["correct"] else "✗"
        print(f"  seed={seed:02d}  predicted={str(r['predicted']):>4}  "
              f"expected={r['expected']}  {status}  raw={repr(r['raw'][:40])}")
        phase_results.append(r)
    all_results[phase_key] = phase_results

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'max_new_tokens', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



───────────────────────────────────────────────────────
  PHASE1_BASELINE  |  10 identical tokens (apple x10)
───────────────────────────────────────────────────────


Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  seed=00  predicted=   8  expected=10  ✗  raw='8'
  seed=01  predicted=   8  expected=10  ✗  raw='8'
  seed=02  predicted=   8  expected=10  ✗  raw='8'
  seed=03  predicted=   8  expected=10  ✗  raw='8'


Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  seed=00  predicted=  10  expected=10  ✓  raw='10'
  seed=01  predicted=  10  expected=10  ✓  raw='10'
  seed=02  predicted=  10  expected=10  ✓  raw='10'


Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  seed=03  predicted=  10  expected=10  ✓  raw='10'
  seed=04  predicted=  10  expected=10  ✓  raw='10'
  seed=05  predicted=  10  expected=10  ✓  raw='10'


Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  seed=06  predicted=  10  expected=10  ✓  raw='10'
  seed=07  predicted=  10  expected=10  ✓  raw='10'
  seed=08  predicted=  10  expected=10  ✓  raw='10'
  seed=09  predicted=  10  expected=10  ✓  raw='10'


## 6 · Summary statistics

In [8]:
print(f"{'Phase':<25} {'Accuracy':>10}  Distribution of predictions")
print(chr(9472) * 65)
for phase_key, results in all_results.items():
    acc   = sum(r["correct"] for r in results) / N_RUNS
    dist  = Counter(str(r["predicted"]) for r in results)
    print(f"{phase_key:<25} {acc:>9.0%}  {dict(dist)}")

# Verify raw outputs
print("\nRaw generation check (first 3 seeds per phase):")
for phase_key, results in all_results.items():
    print(f"\n[{phase_key}]")
    for r in results[:3]:
        print(f"  seed={r['seed']}  raw={repr(r['raw'])}  predicted={r['predicted']}")

Phase                       Accuracy  Distribution of predictions
─────────────────────────────────────────────────────────────────
phase1_baseline                  0%  {'8': 10}
phase2_anomaly                   0%  {'6': 10}
phase3_control                 100%  {'10': 10}

Raw generation check (first 3 seeds per phase):

[phase1_baseline]
  seed=0  raw='8'  predicted=8
  seed=1  raw='8'  predicted=8
  seed=2  raw='8'  predicted=8

[phase2_anomaly]
  seed=0  raw='6'  predicted=6
  seed=1  raw='6'  predicted=6
  seed=2  raw='6'  predicted=6

[phase3_control]
  seed=0  raw='10'  predicted=10
  seed=1  raw='10'  predicted=10
  seed=2  raw='10'  predicted=10


## 7 · Save original results

In [9]:
save_path = f"odometer_results_{MODEL_TAG}.json"
with open(save_path, "w") as f:
    json.dump(all_results, f, indent=2)
print(f"Saved to {save_path}")

Saved to odometer_results_qwen2.5_1.5b_instruct.json


## 8 · Tokenization diagnostic

In [10]:
print("=" * 70)
print("PAYLOAD TOKENIZATION DIAGNOSTIC")
print("=" * 70)

for phase_key, entry in PROMPTS.items():
    payload = entry["text"].split(": ")[1].split(". Respond")[0]
    toks    = tokenizer.encode(payload, add_special_tokens=False)
    decoded = [tokenizer.decode([t]) for t in toks]
    match   = "✓ token count == word count" if len(toks) == len(payload.split()) \
              else f"✗ MISMATCH: {len(toks)} tokens vs {len(payload.split())} words"
    print(f"\n[{phase_key}]")
    print(f"  Payload      : {payload}")
    print(f"  Word count   : {len(payload.split())}")
    print(f"  Token count  : {len(toks)}")
    print(f"  Tokens       : {decoded}")
    print(f"  {match}")

PAYLOAD TOKENIZATION DIAGNOSTIC

[phase1_baseline]
  Payload      : apple apple apple apple apple apple apple apple apple apple
  Word count   : 10
  Token count  : 10
  Tokens       : ['apple', ' apple', ' apple', ' apple', ' apple', ' apple', ' apple', ' apple', ' apple', ' apple']
  ✓ token count == word count

[phase2_anomaly]
  Payload      : apple apple apple apple banana apple apple apple apple apple
  Word count   : 10
  Token count  : 10
  Tokens       : ['apple', ' apple', ' apple', ' apple', ' banana', ' apple', ' apple', ' apple', ' apple', ' apple']
  ✓ token count == word count

[phase3_control]
  Payload      : dog cat car red blue green house tree book pen
  Word count   : 10
  Token count  : 10
  Tokens       : ['dog', ' cat', ' car', ' red', ' blue', ' green', ' house', ' tree', ' book', ' pen']
  ✓ token count == word count


## 9 · Fixed prompts — spot check then run

In [11]:
# Spot check first — if any phase fails here, adjust PROMPTS_FIXED wording
# before running the full fixed-prompt experiment below.
# Common issue: 'comma-separated' in P3 instruction causes overcounting in some models.
# Fix: remove that phrase from the P3 text (already done in PROMPTS_FIXED above).

print("Spot check fixed prompts:")
for phase_key, entry in PROMPTS_FIXED.items():
    raw = pipe(
        [{"role": "user", "content": entry["text"]}],
        max_new_tokens=8, temperature=0.0, do_sample=False,
        pad_token_id=tokenizer.eos_token_id, return_full_text=False,
    )[0]["generated_text"].strip()
    predicted = extract_count(raw)
    correct   = "✓" if predicted == entry["expected"] else "✗"
    print(f"  {phase_key:<25}  expected={entry['expected']}  got={predicted}  {correct}")

print("\nIf any phase shows ✗ above, adjust PROMPTS_FIXED wording before continuing.")

Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Spot check fixed prompts:
  phase1_baseline            expected=10  got=10  ✓
  phase2_anomaly             expected=9  got=8  ✗
  phase3_control             expected=10  got=10  ✓

If any phase shows ✗ above, adjust PROMPTS_FIXED wording before continuing.


In [12]:
# Check original prompts on Qwen2.5-1.5B before deciding what to fix
print("Original space-separated results:")
for phase_key, entry in PROMPTS.items():
    raw = pipe(
        [{"role": "user", "content": entry["text"]}],
        max_new_tokens=8, temperature=0.0, do_sample=False,
        pad_token_id=tokenizer.eos_token_id, return_full_text=False,
    )[0]["generated_text"].strip()
    predicted = extract_count(raw)
    correct   = "✓" if predicted == entry["expected"] else "✗"
    print(f"  {phase_key:<25}  expected={entry['expected']}  got={predicted}  {correct}")

Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Original space-separated results:
  phase1_baseline            expected=10  got=8  ✗
  phase2_anomaly             expected=9  got=6  ✗
  phase3_control             expected=10  got=10  ✓


In [13]:
# Run full fixed-prompt experiment (10 seeds)
all_results_fixed = {}

for phase_key, entry in PROMPTS_FIXED.items():
    print(f"\n{chr(9472)*55}")
    print(f"  {phase_key.upper()} [FIXED]  |  {entry['description']}")
    print(f"{chr(9472)*55}")
    phase_results = []
    for seed in SEEDS:
        torch.manual_seed(seed)
        random.seed(seed)
        np.random.seed(seed)
        raw = pipe(
            [{"role": "user", "content": entry["text"]}],
            max_new_tokens=MAX_NEW_TOKENS, temperature=0.0, do_sample=False,
            pad_token_id=tokenizer.eos_token_id, return_full_text=False,
        )[0]["generated_text"].strip()
        predicted = extract_count(raw)
        correct   = (predicted == entry["expected"]) if predicted is not None else False
        status    = "✓" if correct else "✗"
        print(f"  seed={seed:02d}  predicted={str(predicted):>4}  "
              f"expected={entry['expected']}  {status}")
        phase_results.append({"seed": seed, "raw": raw, "predicted": predicted,
                               "expected": entry["expected"], "correct": correct})
    all_results_fixed[phase_key] = phase_results

# Accuracy comparison
print("ACCURACY COMPARISON: ORIGINAL vs FIXED PROMPTS")
print("=" * 70)
print(f"  {'Phase':<25} {'Original':>10} {'Fixed':>10}  {'Delta':>8}")
print("  " + "-" * 58)
for phase_key in PROMPTS:
    orig  = sum(r["correct"] for r in all_results[phase_key]) / N_RUNS
    fixed = sum(r["correct"] for r in all_results_fixed[phase_key]) / N_RUNS
    print(f"  {phase_key:<25} {orig:>10.0%} {fixed:>10.0%}  {fixed-orig:>+8.0%}")

Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



───────────────────────────────────────────────────────
  PHASE1_BASELINE [FIXED]  |  10 identical tokens, comma-separated
───────────────────────────────────────────────────────
  seed=00  predicted=  10  expected=10  ✓
  seed=01  predicted=  10  expected=10  ✓


Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  seed=02  predicted=  10  expected=10  ✓
  seed=03  predicted=  10  expected=10  ✓
  seed=04  predicted=  10  expected=10  ✓


Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  seed=00  predicted=  10  expected=10  ✓
  seed=01  predicted=  10  expected=10  ✓
  seed=02  predicted=  10  expected=10  ✓


Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  seed=03  predicted=  10  expected=10  ✓
  seed=04  predicted=  10  expected=10  ✓
  seed=05  predicted=  10  expected=10  ✓


Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  seed=06  predicted=  10  expected=10  ✓
  seed=07  predicted=  10  expected=10  ✓
  seed=08  predicted=  10  expected=10  ✓
  seed=09  predicted=  10  expected=10  ✓
ACCURACY COMPARISON: ORIGINAL vs FIXED PROMPTS
  Phase                       Original      Fixed     Delta
  ----------------------------------------------------------
  phase1_baseline                   0%       100%     +100%
  phase2_anomaly                    0%         0%       +0%
  phase3_control                  100%       100%       +0%


## 10 · Behavioral n-sweep

In [14]:
unique_vocab = ["dog", "cat", "car", "red", "blue", "green",
                "house", "tree", "book", "pen", "fish", "cup",
                "hat", "sun", "moon", "sky", "fire", "rain", "snow", "wind"]

NS = [5, 6, 7, 8, 9, 10, 11, 12, 15, 20]

print("=" * 65)
print(f"BEHAVIORAL N-SWEEP — {MODEL_NAME}")
print("=" * 65)
print(f"  {'n':>4}  {'P1 output':>10}  {'P1 correct':>11}  "
      f"{'P3 output':>10}  {'P3 correct':>11}")
print("  " + "-" * 55)

sweep_behavioral = {}

for n in NS:
    prompt_p1 = (
        f'Count the number of times "apple" appears in this list: '
        + " ".join(["apple"] * n)
        + ". Respond only with the integer, nothing else."
    )
    prompt_p3 = (
        "Count the number of words in this list: "
        + " ".join(unique_vocab[:n])
        + ". Respond only with the integer, nothing else."
    )
    raw_p1 = pipe([{"role": "user", "content": prompt_p1}],
                  max_new_tokens=8, temperature=0.0, do_sample=False,
                  pad_token_id=tokenizer.eos_token_id,
                  return_full_text=False)[0]["generated_text"].strip()
    raw_p3 = pipe([{"role": "user", "content": prompt_p3}],
                  max_new_tokens=8, temperature=0.0, do_sample=False,
                  pad_token_id=tokenizer.eos_token_id,
                  return_full_text=False)[0]["generated_text"].strip()

    pred_p1 = extract_count(raw_p1)
    pred_p3 = extract_count(raw_p3)
    corr_p1 = "✓" if pred_p1 == n else "✗"
    corr_p3 = "✓" if pred_p3 == n else "✗"
    print(f"  {n:>4}  {str(pred_p1):>10}  {corr_p1:>11}  "
          f"{str(pred_p3):>10}  {corr_p3:>11}")
    sweep_behavioral[n] = {
        "n": n,
        "p1_output": pred_p1, "p1_correct": pred_p1 == n,
        "p3_output": pred_p3, "p3_correct": pred_p3 == n,
    }

print("\nP1 attractor pattern:")
for n, r in sweep_behavioral.items():
    marker = "= n (correct)" if r["p1_correct"] else f"≠ n (wrong, got {r['p1_output']})"
    print(f"  n={n:>2}  output={r['p1_output']}  {marker}")

with open(f"behavioral_n_sweep_{MODEL_TAG}.json", "w") as f:
    json.dump({str(k): v for k, v in sweep_behavioral.items()}, f, indent=2)
print(f"\nSaved: behavioral_n_sweep_{MODEL_TAG}.json")

Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


BEHAVIORAL N-SWEEP — Qwen/Qwen2.5-1.5B-Instruct
     n   P1 output   P1 correct   P3 output   P3 correct
  -------------------------------------------------------
     5           5            ✓           5            ✓


Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


     6           6            ✓           5            ✗
     7           7            ✓           6            ✗


Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

    10           8            ✗          10            ✓
    11          10            ✗          10            ✗


Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    12          10            ✗          12            ✓
    15          10            ✗          12            ✗


Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    20          10            ✗          17            ✗

P1 attractor pattern:
  n= 5  output=5  = n (correct)
  n= 6  output=6  = n (correct)
  n= 7  output=7  = n (correct)
  n= 8  output=8  = n (correct)
  n= 9  output=8  ≠ n (wrong, got 8)
  n=10  output=8  ≠ n (wrong, got 8)
  n=11  output=10  ≠ n (wrong, got 10)
  n=12  output=10  ≠ n (wrong, got 10)
  n=15  output=10  ≠ n (wrong, got 10)
  n=20  output=10  ≠ n (wrong, got 10)

Saved: behavioral_n_sweep_qwen2.5_1.5b_instruct.json


## 11 · Prompt paraphrase robustness

In [15]:
PARAPHRASES = {
    "original": (
        'Count the number of times "apple" appears in this list: '
        "{list}. Respond only with the integer, nothing else."
    ),
    "how_many": (
        'How many times does the word "apple" appear in the following list: '
        "{list}? Answer with a single integer, nothing else."
    ),
    "list_first": (
        "List: {list}\n"
        'How many times does "apple" appear? Single integer only.'
    ),
    "tally": (
        'Tally the occurrences of "apple" in this sequence: '
        "{list}. Output only the count as an integer."
    ),
    "simple": (
        'Count "apple" in: {list}. '
        "Reply with just the number."
    ),
}

APPLE_LIST_10         = " ".join(["apple"] * 10)
APPLE_LIST_10_ANOMALY = "apple apple apple apple banana apple apple apple apple apple"
UNIQUE_LIST_10        = "dog cat car red blue green house tree book pen"

test_cases = {
    "P1_repeated": (APPLE_LIST_10,         10),
    "P2_anomaly" : (APPLE_LIST_10_ANOMALY,  9),
    "P3_unique"  : (UNIQUE_LIST_10,         10),
}

print("=" * 75)
print(f"PROMPT PARAPHRASE ROBUSTNESS — {MODEL_NAME}")
print("=" * 75)
print(f"  {'Paraphrase':<15}", end="")
for case in test_cases:
    print(f"  {case:<16}", end="")
print()
print("  " + "-" * 68)

paraphrase_results = {}

for pname, template in PARAPHRASES.items():
    print(f"  {pname:<15}", end="")
    paraphrase_results[pname] = {}
    for case_name, (word_list, expected) in test_cases.items():
        if case_name == "P3_unique":
            prompt = ("Count the number of words in this list: "
                      f"{word_list}. Respond only with the integer, nothing else.")
        else:
            prompt = template.format(list=word_list)
        raw = pipe([{"role": "user", "content": prompt}],
                   max_new_tokens=8, temperature=0.0, do_sample=False,
                   pad_token_id=tokenizer.eos_token_id,
                   return_full_text=False)[0]["generated_text"].strip()
        predicted = extract_count(raw)
        correct   = "✓" if predicted == expected else "✗"
        print(f"  {str(predicted)+'('+correct+')' :<16}", end="")
        paraphrase_results[pname][case_name] = {
            "predicted": predicted, "expected": expected,
            "correct": predicted == expected,
        }
    print()

print("\nP1 attractor stability:")
p1_preds = [paraphrase_results[p]["P1_repeated"]["predicted"] for p in PARAPHRASES]
modal    = Counter(p1_preds).most_common(1)[0][0]
for pname in PARAPHRASES:
    pred = paraphrase_results[pname]["P1_repeated"]["predicted"]
    tag  = "STABLE" if pred == modal else f"SHIFTED from {modal}"
    print(f"  {pname:<15}  predicted={pred}  {tag}")

with open(f"paraphrase_robustness_{MODEL_TAG}.json", "w") as f:
    json.dump(paraphrase_results, f, indent=2)
print(f"\nSaved: paraphrase_robustness_{MODEL_TAG}.json")

Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT PARAPHRASE ROBUSTNESS — Qwen/Qwen2.5-1.5B-Instruct
  Paraphrase       P1_repeated       P2_anomaly        P3_unique       
  --------------------------------------------------------------------
  original         8(✗)              6(✗)            

Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  10(✓)           
  how_many         8(✗)              6(✗)              10(✓)           
  list_first     

Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  10(✓)             10(✗)             10(✓)           
  tally          

Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  10(✓)           

Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  None(✗)           10(✓)           
  simple           8(✗)              6(✗)              10(✓)           

P1 attractor stability:
  original         predicted=8  STABLE
  how_many         predicted=8  STABLE
  list_first       predicted=10  SHIFTED from 8
  tally            predicted=10  SHIFTED from 8
  simple           predicted=8  STABLE

Saved: paraphrase_robustness_qwen2.5_1.5b_instruct.json


## 12 · Load model_eager (for mechanistic cells)

Keep `model` (sdpa) alive for inference — it's faster.
`model_eager` is only needed for attention extraction and logit lens.

In [16]:
model_eager = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)
model_eager.eval()
print(f"Eager model loaded.")
print(f"Layers     : {model_eager.config.num_hidden_layers}")
print(f"Attn impl  : {model_eager.config._attn_implementation}")

# Update make_inputs to use model_eager device for mechanistic cells
def make_inputs_eager(prompt_text):
    messages = [{"role": "user", "content": prompt_text}]
    return tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model_eager.device)

# Inspect layer submodule names — needed to confirm hook targets
# Qwen2.5 should match Llama (self_attn, mlp, input_layernorm, post_attention_layernorm)
# but verify here in case of version differences
print("\nLayer submodule names:")
for name, _ in model_eager.model.layers[0].named_children():
    print(f"  {name}")
print("\nExpected: self_attn, mlp, input_layernorm, post_attention_layernorm")
print("If different, update hook targets in cells 15 and 16.")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Eager model loaded.
Layers     : 28
Attn impl  : eager

Layer submodule names:
  self_attn
  mlp
  input_layernorm
  post_attention_layernorm

Expected: self_attn, mlp, input_layernorm, post_attention_layernorm
If different, update hook targets in cells 15 and 16.


## 13 · Attention analysis

In [17]:
def get_attentions(prompt_text):
    inputs = make_inputs_eager(prompt_text)
    with torch.no_grad():
        out = model_eager(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            output_attentions=True,
        )
    attentions = torch.stack(out.attentions).squeeze(1)  # (n_layers, n_heads, seq, seq)
    return attentions, inputs["input_ids"][0]


def find_word_positions(tokens, phase_key):
    content_words = {
        "phase1_baseline": {"apple"},
        "phase2_anomaly" : {"apple", "banana"},
        "phase3_control" : {"dog", "cat", "car", "red", "blue",
                            "green", "house", "tree", "book", "pen"},
    }
    valid = content_words[phase_key]
    colon_idx = max(i for i, t in enumerate(tokens) if t.strip() == ":")
    positions = []
    for i in range(colon_idx + 1, len(tokens)):
        if tokens[i].strip() in valid:
            positions.append(i)
        elif "." in tokens[i] and positions:
            break
    return positions


print("=" * 70)
print("ATTENTION ANALYSIS — word-list tokens only")
print("=" * 70)

attn_summary = {}

for phase_key in ["phase1_baseline", "phase2_anomaly", "phase3_control"]:
    attentions, input_ids = get_attentions(PROMPTS[phase_key]["text"])
    tokens        = [tokenizer.decode([t]) for t in input_ids]
    word_positions = find_word_positions(tokens, phase_key)
    word_tokens    = [tokens[p] for p in word_positions]

    per_head  = attentions.cpu().float().numpy()            # (n_layers, n_heads, seq, seq)
    word_attn = per_head[:, :, -1, :][:, :, word_positions] # last token -> word positions
    word_attn_norm = word_attn / (word_attn.sum(axis=-1, keepdims=True) + 1e-9)
    mean_word_attn = word_attn_norm.mean(axis=1)             # (n_layers, n_words)

    entropies = []
    for l in range(mean_word_attn.shape[0]):
        a = mean_word_attn[l] / (mean_word_attn[l].sum() + 1e-9)
        entropies.append(round(float(-(a * np.log(a + 1e-9)).sum()), 4))

    uniformity = (mean_word_attn.min(axis=-1) /
                  (mean_word_attn.max(axis=-1) + 1e-9)).mean()

    print(f"\n[{phase_key}]")
    print(f"  Word positions : {word_positions}")
    print(f"  Word tokens    : {word_tokens}")
    print(f"  Mean entropy   : {np.mean(entropies):.4f}")
    print(f"  Uniformity     : {float(uniformity):.4f}")

    attn_summary[phase_key] = {
        "word_positions": word_positions, "word_tokens": word_tokens,
        "entropy_per_layer": entropies, "mean_uniformity": float(uniformity),
    }

with open(f"attention_analysis_{MODEL_TAG}.json", "w") as f:
    json.dump(attn_summary, f, indent=2)
print(f"\nSaved: attention_analysis_{MODEL_TAG}.json")

ATTENTION ANALYSIS — word-list tokens only

[phase1_baseline]
  Word positions : [37, 38, 39, 40, 41, 42, 43, 44, 45, 46]
  Word tokens    : [' apple', ' apple', ' apple', ' apple', ' apple', ' apple', ' apple', ' apple', ' apple', ' apple']
  Mean entropy   : 2.0452
  Uniformity     : 0.1837

[phase2_anomaly]
  Word positions : [37, 38, 39, 40, 41, 42, 43, 44, 45, 46]
  Word tokens    : [' apple', ' apple', ' apple', ' apple', ' banana', ' apple', ' apple', ' apple', ' apple', ' apple']
  Mean entropy   : 2.0146
  Uniformity     : 0.1383

[phase3_control]
  Word positions : [33, 34, 35, 36, 37, 38, 39, 40, 41, 42]
  Word tokens    : [' dog', ' cat', ' car', ' red', ' blue', ' green', ' house', ' tree', ' book', ' pen']
  Mean entropy   : 2.0600
  Uniformity     : 0.1507

Saved: attention_analysis_qwen2.5_1.5b_instruct.json


## 14 · Linear probes

In [18]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import mean_absolute_error, r2_score


def get_hidden_states(prompt_text):
    inputs = make_inputs_eager(prompt_text)
    with torch.no_grad():
        out = model_eager(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            output_hidden_states=True,
        )
    hidden = torch.stack(out.hidden_states)  # (n_layers+1, 1, seq_len, hidden_dim)
    return hidden[:, 0, -1, :].cpu().float().numpy()  # (n_layers+1, hidden_dim)


def run_loo_probe(hidden_array, labels):
    loo  = LeaveOneOut()
    maes, r2s = [], []
    for layer_idx in range(hidden_array.shape[1]):
        X      = StandardScaler().fit_transform(hidden_array[:, layer_idx, :])
        preds  = np.zeros(len(labels))
        for train_idx, test_idx in loo.split(X):
            clf = Ridge(alpha=1.0)
            clf.fit(X[train_idx], labels[train_idx])
            preds[test_idx] = clf.predict(X[test_idx])
        maes.append(mean_absolute_error(labels, preds))
        r2s.append(r2_score(labels, preds))
    return maes, r2s


probe_ns    = list(range(3, 14))
labels      = np.array(probe_ns, dtype=float)

repeated_prompts = [
    f'Count the number of times "apple" appears in this list: '
    + " ".join(["apple"] * n)
    + ". Respond only with the integer, nothing else."
    for n in probe_ns
]
unique_prompts = [
    "Count the number of words in this list: "
    + " ".join(unique_vocab[:n])
    + ". Respond only with the integer, nothing else."
    for n in probe_ns
]

print("Collecting activations...")
hidden_repeated = np.stack([get_hidden_states(p) for p in repeated_prompts])
hidden_unique   = np.stack([get_hidden_states(p) for p in unique_prompts])

print("Running probes...")
maes_rep,  r2s_rep  = run_loo_probe(hidden_repeated, labels)
maes_uniq, r2s_uniq = run_loo_probe(hidden_unique,   labels)

print(f"\n{'Layer':>6}  {'MAE(rep)':>10}  {'R2(rep)':>9}  "
      f"{'MAE(uniq)':>10}  {'R2(uniq)':>9}  {'ΔMAE':>8}")
print("-" * 62)
n_layers_plus1 = hidden_repeated.shape[1]
for i in range(n_layers_plus1):
    label = "embed" if i == 0 else f"L{i:02d}"
    delta = maes_rep[i] - maes_uniq[i]
    print(f"{label:>6}  {maes_rep[i]:>10.4f}  {r2s_rep[i]:>9.4f}  "
          f"{maes_uniq[i]:>10.4f}  {r2s_uniq[i]:>9.4f}  {delta:>+8.4f}")

probe_results = {
    "ns": probe_ns, "labels": probe_ns,
    "repeated": {"maes": maes_rep,  "r2s": r2s_rep},
    "unique"  : {"maes": maes_uniq, "r2s": r2s_uniq},
}
with open(f"probe_results_{MODEL_TAG}.json", "w") as f:
    json.dump(probe_results, f, indent=2)
print(f"\nSaved: probe_results_{MODEL_TAG}.json")

Running probes...

 Layer    MAE(rep)    R2(rep)   MAE(uniq)   R2(uniq)      ΔMAE
--------------------------------------------------------------
 embed      3.0000    -0.2100      3.0000    -0.2100   +0.0000
   L01      0.8342     0.9023      1.1863     0.7457   -0.3520
   L02      1.2315     0.7336      1.5103     0.5150   -0.2788
   L03      1.3739     0.7253      1.1317     0.7556   +0.2422
   L04      1.3194     0.7069      0.8274     0.8285   +0.4921
   L05      1.1719     0.7687      0.7768     0.8569   +0.3951
   L06      1.3072     0.7239      0.7244     0.8769   +0.5828
   L07      1.3564     0.6874      0.7938     0.8632   +0.5626
   L08      1.3389     0.7076      0.8448     0.8470   +0.4940
   L09      1.1035     0.8044      0.7284     0.8715   +0.3751
   L10      1.1011     0.8017      0.8094     0.8655   +0.2917
   L11      0.9246     0.8538      0.7696     0.8823   +0.1551
   L12      0.8126     0.8988      0.8203     0.8925   -0.0076
   L13      0.8565     0.8924      0

## 15 · Logit lens

**After running this cell:**
1. Identify the layer where the wrong answer locks in for P1
2. Go back to cell 1 and set `CRITICAL_LAYERS` to that layer ± 2
3. Then run cells 16 and 17

In [19]:
def logit_lens_full(prompt_text, top_k=5):
    inputs = make_inputs_eager(prompt_text)
    with torch.no_grad():
        out = model_eager(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            output_hidden_states=True,
        )
    unembed = model_eager.lm_head.weight
    norm    = model_eager.model.norm
    results = []
    for layer_idx, h in enumerate(out.hidden_states):
        last        = h[0, -1, :]
        last_normed = norm(last.unsqueeze(0).unsqueeze(0)).squeeze()
        logits      = unembed @ last_normed
        top_digit   = get_top_digit(logits)
        top5        = [tokenizer.decode([i]) for i in logits.topk(top_k).indices]
        results.append({
            "layer"     : "embed" if layer_idx == 0 else f"L{layer_idx:02d}",
            "top_digit" : top_digit,
            "top5"      : top5,
        })
    return results


print("=" * 70)
print(f"LOGIT LENS — {MODEL_NAME}")
print(f"Total layers: {model_eager.config.num_hidden_layers}")
print("=" * 70)

lens_results = {}

for phase_key in ["phase1_baseline", "phase3_control"]:
    correct = PROMPTS[phase_key]["expected"]
    print(f"\n[{phase_key}]  correct={correct}")
    print(f"  {'Layer':>6}  {'Top digit':>10}  Top-5 tokens")
    print("  " + "-" * 55)
    lens = logit_lens_full(PROMPTS[phase_key]["text"])
    lens_results[phase_key] = lens
    for r in lens:
        print(f"  {r['layer']:>6}  {r['top_digit']:>10}  {r['top5']}")

# Normalized depth summary
n_layers = model_eager.config.num_hidden_layers
print(f"\n{'='*50}")
print("LOCK-IN SUMMARY")
print(f"{'='*50}")
print(f"Total layers: {n_layers}")
print(f"Equivalent of 87.5% depth (Llama-1B lock-in): L{round(0.875 * n_layers)}")
print("\n>>> After reading P1 output above:")
print(">>> Go to cell 1, set CRITICAL_LAYERS = [lock_in_layer-2 .. lock_in_layer+2]")
print(">>> Then run cells 16 and 17")

with open(f"logit_lens_{MODEL_TAG}.json", "w") as f:
    json.dump(lens_results, f, indent=2)
print(f"\nSaved: logit_lens_{MODEL_TAG}.json")

LOGIT LENS — Qwen/Qwen2.5-1.5B-Instruct
Total layers: 28

[phase1_baseline]  correct=10
   Layer   Top digit  Top-5 tokens
  -------------------------------------------------------
   embed           1  ['\n', '\n\n', ' ', ',', '\r\n']
     L01           1  ['s', '<<<<<<<', 'with', 'sign', 'for']
     L02           1  ['s', '<<<<<<<', 'with', ' ', ' with']
     L03           1  ['s', '<<<<<<<', ' -', '+', ' ']
     L04           1  ['<<<<<<<', 's', '<<', 'https', 'skip']
     L05           1  ['s', '<<<<<<<', ' I', 'with', '   ']
     L06           1  ['s', '<<<<<<<', ' Mode', '@', '```']
     L07           1  ['s', '<<<<<<<', ' E', 'E', '**']
     L08           1  ['s', 'is', '**', ' A', 'E']
     L09           1  ['s', 'to', ' P', ' B', '<<<<<<<']
     L10           1  ['s', ' B', ' ', 'to', 'B']
     L11           1  ['s', ' ', '   ', '```', 'to']
     L12           1  ['s', 'to', 'Ass', '```', ' Ass']
     L13           1  [' ', '<|endoftext|>', ' True', '1', 'Ass']
     L14       

## 16 · MLP vs attention decomposition

**Requires CRITICAL_LAYERS to be set in cell 1 after running logit lens.**

Hook target is `post_attention_layernorm` — its input is the residual stream
after attention addition and before MLP. If the layer inspect in cell 12
showed different submodule names, update `layer.post_attention_layernorm` below.

In [21]:
assert CRITICAL_LAYERS is not None, \
    "Set CRITICAL_LAYERS in cell 1 after reading logit lens output (cell 15)"


def logit_lens_single(hidden_state):
    norm    = model_eager.model.norm
    unembed = model_eager.lm_head.weight
    with torch.no_grad():
        normed = norm(hidden_state.unsqueeze(0).unsqueeze(0)).squeeze()
        logits = unembed @ normed
    top_digit = get_top_digit(logits)
    top5      = [tokenizer.decode([i]) for i in logits.topk(5).indices]
    return top_digit, top5


def decompose_layer(prompt_text, layer_idx):
    """
    Extract residual stream at three points around layer_idx:
      h_before    : entering the decoder layer
      h_post_attn : after attention addition, before MLP
                    (= input to post_attention_layernorm)
      h_post_layer: after full layer (attn + MLP)

    Hook target for h_post_attn is post_attention_layernorm.
    If Qwen uses a different name (check cell 12 output),
    replace 'post_attention_layernorm' with the correct name.
    """
    inputs = make_inputs_eager(prompt_text)
    cache  = {}
    layer  = model_eager.model.layers[layer_idx - 1]

    def hook_pre_layer(module, input):
        h = input[0] if isinstance(input, tuple) else input
        if isinstance(h, torch.Tensor):
            cache["h_before"] = h[0, -1, :].detach().clone()

    def hook_post_attn(module, input, output):
        h = input[0] if isinstance(input, tuple) else input
        if isinstance(h, torch.Tensor):
            cache["h_post_attn"] = h[0, -1, :].detach().clone()

    def hook_post_layer(module, input, output):
        h = output[0] if isinstance(output, tuple) else output
        if isinstance(h, torch.Tensor):
            cache["h_post_layer"] = h[0, -1, :].detach().clone()

    h1 = layer.register_forward_pre_hook(hook_pre_layer)
    # ── If cell 12 showed different name, change 'post_attention_layernorm' here
    h2 = layer.post_attention_layernorm.register_forward_hook(hook_post_attn)
    h3 = layer.register_forward_hook(hook_post_layer)

    with torch.no_grad():
        model_eager(**inputs)

    h1.remove(); h2.remove(); h3.remove()

    if len(cache) < 3:
        print(f"  Warning L{layer_idx}: only captured {list(cache.keys())}")

    return {
        name: {"top_digit": logit_lens_single(h)[0],
               "top5"     : logit_lens_single(h)[1]}
        for name, h in cache.items()
    }


decomp_results = {}

for phase_key in ["phase1_baseline", "phase3_control"]:
    # Determine wrong answer for this phase from behavioral results
    wrong_answer = str(all_results[phase_key][0]["predicted"])

    print(f"\n{'='*72}")
    print(f"MLP vs ATTENTION DECOMPOSITION — {phase_key}")
    print(f"Wrong answer to track: '{wrong_answer}'")
    print(f"{'='*72}")
    print(f"  {'Layer':>6}  {'Before':>10}  {'Post-attn':>10}  "
          f"{'Post-MLP':>10}  Writer")
    print("  " + "-" * 58)

    for layer_idx in CRITICAL_LAYERS:
        remove_all_hooks(model_eager)
        r = decompose_layer(PROMPTS[phase_key]["text"], layer_idx)

        before    = r.get("h_before",    {}).get("top_digit", "?")
        post_attn = r.get("h_post_attn", {}).get("top_digit", "?")
        post_mlp  = r.get("h_post_layer",{}).get("top_digit", "?")

        if before != wrong_answer and post_attn == wrong_answer:
            writer = "ATTENTION"
        elif post_attn != wrong_answer and post_mlp == wrong_answer:
            writer = "MLP"
        elif before == wrong_answer:
            writer = "(already)"
        else:
            writer = "-"

        print(f"  L{layer_idx:02d}    {before:>10}  {post_attn:>10}  "
              f"{post_mlp:>10}  {writer}")
        decomp_results[f"{phase_key}_L{layer_idx:02d}"] = {
            "phase": phase_key, "layer": layer_idx,
            "h_before"   : r.get("h_before",    {}),
            "h_post_attn": r.get("h_post_attn", {}),
            "h_post_layer": r.get("h_post_layer",{}),
        }

with open(f"mlp_attn_decomp_{MODEL_TAG}.json", "w") as f:
    json.dump(decomp_results, f, indent=2)
print(f"\nSaved: mlp_attn_decomp_{MODEL_TAG}.json")


MLP vs ATTENTION DECOMPOSITION — phase1_baseline
Wrong answer to track: '8'
   Layer      Before   Post-attn    Post-MLP  Writer
  ----------------------------------------------------------
  L20             1           1           1  -
  L21             1           7           4  -
  L22             4           4           8  MLP
  L23             8           7           7  (already)
  L24             7           7           8  MLP
  L25             8           8           8  (already)
  L26             8           8           8  (already)

MLP vs ATTENTION DECOMPOSITION — phase3_control
Wrong answer to track: '10'
   Layer      Before   Post-attn    Post-MLP  Writer
  ----------------------------------------------------------
  L20             6           6           7  -
  L21             7           7           7  -
  L22             7           7           7  -
  L23             7           7           9  -
  L24             9           9           5  -
  L25             5       

## 17 · Per-n MLP decomposition at lock-in layer

Checks whether the MLP at the lock-in layer writes the wrong answer
consistently across n values. 

Set `LOCKIN_LAYER` to the layer identified
as the MLP writer in cell 16.

In [22]:
# ── Set this to the MLP writer layer found in cell 16 ─────────────────────
# Example: if cell 16 showed MLP writes wrong answer at L22, set LOCKIN_LAYER = 22
LOCKIN_LAYER = CRITICAL_LAYERS[len(CRITICAL_LAYERS) // 2]  # placeholder: middle of range
# Replace with the actual writer layer after reading cell 16 output

wrong_answer = str(all_results["phase1_baseline"][0]["predicted"])

print(f"Per-n MLP decomposition at L{LOCKIN_LAYER}")
print(f"Tracking wrong answer: '{wrong_answer}'")
print(f"{'n':>4}  {'Before':>10}  {'Post-attn':>10}  "
      f"{'Post-MLP':>10}  {'MLP wrote wrong?':>18}")
print("-" * 60)

pern_results = {}

for n in [7, 8, 9, 10, 11, 12, 15]:
    prompt = (
        f'Count the number of times "apple" appears in this list: '
        + " ".join(["apple"] * n)
        + ". Respond only with the integer, nothing else."
    )
    remove_all_hooks(model_eager)
    r = decompose_layer(prompt, LOCKIN_LAYER)

    before    = r.get("h_before",    {}).get("top_digit", "?")
    post_attn = r.get("h_post_attn", {}).get("top_digit", "?")
    post_mlp  = r.get("h_post_layer",{}).get("top_digit", "?")

    wrote_wrong = (
        "YES" if post_attn != wrong_answer and post_mlp == wrong_answer
        else "already" if before == wrong_answer
        else "no"
    )
    print(f"{n:>4}  {before:>10}  {post_attn:>10}  "
          f"{post_mlp:>10}  {wrote_wrong:>18}")
    pern_results[n] = {
        "n": n, "before": before, "post_attn": post_attn,
        "post_mlp": post_mlp, "wrote_wrong": wrote_wrong,
    }

with open(f"pern_mlp_decomp_{MODEL_TAG}.json", "w") as f:
    json.dump({str(k): v for k, v in pern_results.items()}, f, indent=2)
print(f"\nSaved: pern_mlp_decomp_{MODEL_TAG}.json")

Per-n MLP decomposition at L23
Tracking wrong answer: '8'
   n      Before   Post-attn    Post-MLP    MLP wrote wrong?
------------------------------------------------------------
   7           7           7           7                  no
   8           8           7           7             already
   9           8           7           7             already
  10           8           7           7             already
  11           8           7           7             already
  12           8           7           7             already
  15           8           7           7             already

Saved: pern_mlp_decomp_qwen2.5_1.5b_instruct.json


## 18 · Save all results summary

In [24]:
summary = {
    "model"       : MODEL_NAME,
    "model_tag"   : MODEL_TAG,
    "n_layers"    : model_eager.config.num_hidden_layers,
    "hidden_size" : model_eager.config.hidden_size,
    "behavioral": {
        phase_key: {
            "accuracy_original": sum(r["correct"] for r in all_results[phase_key]) / N_RUNS,
            "accuracy_fixed"   : sum(r["correct"] for r in all_results_fixed[phase_key]) / N_RUNS,
            "attractor"        : all_results[phase_key][0]["predicted"],
        }
        for phase_key in PROMPTS
    },
    "critical_layers": CRITICAL_LAYERS,
    "lockin_layer"   : LOCKIN_LAYER if 'LOCKIN_LAYER' in dir() else None,
}

with open(f"summary_{MODEL_TAG}.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"FINAL SUMMARY — {MODEL_NAME}")
print(f"{'='*55}")
print(f"  Layers     : {summary['n_layers']}")
print(f"  Hidden dim : {summary['hidden_size']}")
print()
print(f"  {'Phase':<25} {'Orig':>6}  {'Fixed':>6}  {'Attractor':>10}")
print("  " + "-" * 52)
for phase_key, b in summary["behavioral"].items():
    print(f"  {phase_key:<25} {b['accuracy_original']:>6.0%}  "
          f"{b['accuracy_fixed']:>6.0%}  {str(b['attractor']):>10}")
print(f"\nSaved: summary_{MODEL_TAG}.json")

FINAL SUMMARY — Qwen/Qwen2.5-1.5B-Instruct
  Layers     : 28
  Hidden dim : 1536

  Phase                       Orig   Fixed   Attractor
  ----------------------------------------------------
  phase1_baseline               0%    100%           8
  phase2_anomaly                0%      0%           6
  phase3_control              100%    100%          10

Saved: summary_qwen2.5_1.5b_instruct.json


In [26]:
# Save Qwen 1.5B summary
summary_qwen15 = {
    "model"          : MODEL_NAME,
    "n_layers"       : 28,
    "p1_attractor"   : "8",
    "p2_attractor"   : "6",
    "p3_correct"     : True,
    "comma_fixes_p1" : True,
    "comma_fixes_p2" : False,
    "n_sweep_pattern": "correct n<=8, '8' for n=9-10, '10' for n=11+",
    "probe_note"     : "repeated tokens harder to probe than unique in early layers — opposite of Llama 1B",
    "mlp_writer"     : "L22 and L24 — distributed, same depth as Llama 3B L22",
    "lock_in_depth"  : round(22/28, 3),
    "paraphrase_stable": "original, how_many, simple",
    "paraphrase_shifted": "list_first->10(correct), tally->10(correct)",
    "cross_family_notes": [
        "P1 attractor '8' matches Llama 1B",
        "P2 attractor '6' is unique — banana shifts by -2, not seen in Llama",
        "P3 correct matches Llama 1B",
        "Probe R2 for repeated tokens reaches 0.97 only at L19+ vs Llama 1B L01",
        "Layer submodule names identical to Llama — hooks work unchanged"
    ]
}
with open("complete_summary_qwen2.5_1.5b.json", "w") as f:
    json.dump(summary_qwen15, f, indent=2)
print("Saved: complete_summary_qwen2.5_1.5b.json")

Saved: complete_summary_qwen2.5_1.5b.json


## Diagnostic Check

Run all three diagnostic tests for **Qwen2.5-1.5B** while for 3B and 7B, run tokenizer check (Diagnostic 2) ONLY.

In [30]:
# ── Diagnostic 1: Probe dissociation check for Qwen ──────────────────────
# Is the count encoded but not used (like Llama 1B)?
# Or is it simply not encoded until late layers?
# Key question: at the layers where the wrong answer locks in (L22-L25),
# is the probe R2 high enough to claim dissociation?

import json
import numpy as np

with open("qwen1.5B/probe_results_qwen2.5_1.5b_instruct.json") as f:
    probe_data = json.load(f)

maes_rep  = probe_data["repeated"]["maes"]
r2s_rep   = probe_data["repeated"]["r2s"]
maes_uniq = probe_data["unique"]["maes"]
r2s_uniq  = probe_data["unique"]["r2s"]

print("=" * 65)
print("PROBE DISSOCIATION DIAGNOSTIC — Qwen2.5-1.5B")
print("=" * 65)
print(f"\nKey question: is R2(repeated) high at the lock-in layers (L22-L25)?")
print(f"Lock-in layers for Qwen 1.5B: L22 (MLP writes '8'), L24 (MLP writes '8')")
print(f"\n{'Layer':>6}  {'R2(rep)':>9}  {'R2(uniq)':>9}  {'MAE(rep)':>9}  "
      f"{'MAE(uniq)':>10}  {'Dissociation?':>15}")
print("-" * 70)

LOCKIN_LAYERS = [20, 21, 22, 23, 24, 25, 26]
DISSOC_THRESHOLD = 0.95  # minimum R2 to claim count is encoded

for i in range(len(maes_rep)):
    label    = "embed" if i == 0 else f"L{i:02d}"
    r2r      = r2s_rep[i]
    r2u      = r2s_uniq[i]
    mar      = maes_rep[i]
    mau      = maes_uniq[i]
    is_lockin = i in LOCKIN_LAYERS
    # Dissociation = count encoded (R2 high) but model outputs wrong answer
    dissoc   = "YES" if r2r > DISSOC_THRESHOLD and is_lockin \
               else "weak" if r2r > 0.90 and is_lockin \
               else "NO" if is_lockin \
               else "-"
    marker   = " <-- LOCK-IN" if is_lockin else ""
    print(f"{label:>6}  {r2r:>9.4f}  {r2u:>9.4f}  {mar:>9.4f}  "
          f"{mau:>10.4f}  {dissoc:>15}{marker}")

# Summary
print(f"\nDissociation summary:")
lockin_r2s = [r2s_rep[i] for i in LOCKIN_LAYERS]
print(f"  Mean R2(repeated) at lock-in layers L20-L26: {np.mean(lockin_r2s):.4f}")
print(f"  Min  R2(repeated) at lock-in layers L20-L26: {np.min(lockin_r2s):.4f}")
print(f"  Llama 1B R2 at lock-in (L14): {0.9945:.4f}  (for reference)")
print(f"  Threshold for claiming dissociation: {DISSOC_THRESHOLD}")

if np.mean(lockin_r2s) > DISSOC_THRESHOLD:
    print(f"\n  VERDICT: Dissociation holds for Qwen — count encoded at lock-in layers")
elif np.mean(lockin_r2s) > 0.90:
    print(f"\n  VERDICT: Weak dissociation — count partially encoded at lock-in layers")
else:
    print(f"\n  VERDICT: No dissociation — count not encoded at lock-in layers")


# ── Diagnostic 2: Logit lens tokenizer check ─────────────────────────────
# Why does P3 output "10" correctly but "10" never appears in logit lens?
# Check: is "10" a single token in Qwen tokenizer?
# If "10" tokenizes to ["1", "0"] then get_top_digit skips it entirely.

print("\n" + "=" * 65)
print("LOGIT LENS TOKENIZER DIAGNOSTIC — Qwen2.5-1.5B")
print("=" * 65)
print("\nChecking digit tokenization (single token = usable in logit lens):")
print(f"{'Digit':>6}  {'Token IDs':>20}  {'Decoded':>15}  Single token?")
print("-" * 58)

for n in range(1, 21):
    ids     = tokenizer.encode(str(n), add_special_tokens=False)
    decoded = [tokenizer.decode([i]) for i in ids]
    single  = "YES" if len(ids) == 1 else f"NO — {len(ids)} tokens"
    print(f"{n:>6}  {str(ids):>20}  {str(decoded):>15}  {single}")

# This directly shows which digits the logit lens can track
# If "10" is multi-token, it explains why P3 logit lens never shows "10"
# despite the model outputting it correctly

print("\nImplication for logit lens:")
ids_10 = tokenizer.encode("10", add_special_tokens=False)
if len(ids_10) > 1:
    print(f"  '10' tokenizes to {ids_10} — multi-token, invisible to logit lens")
    print(f"  The logit lens top-digit tracking MISSES '10' entirely")
    print(f"  P3 outputs '10' correctly but logit lens can't show it")
    print(f"  This is a methodological limitation, not a model failure")
    print(f"  Logit lens results for Qwen should only be interpreted for")
    print(f"  single-token digits: {[n for n in range(1,21) if len(tokenizer.encode(str(n), add_special_tokens=False))==1]}")
else:
    print(f"  '10' is a single token — logit lens should track it")
    print(f"  P3 failure to show '10' is a genuine model behavior, not an artifact")

# ── Diagnostic 3: Direct logit check at final layer for P3 ───────────────
# What are the actual logits for "10" at the final layer of P3?
# Even if not top-1, where does "10" rank?

print("DIRECT LOGIT CHECK — P3 final layer, Qwen2.5-1.5B")
print("=" * 65)

remove_all_hooks(model_eager)

inputs = make_inputs_eager(PROMPTS["phase3_control"]["text"])

# Collect final hidden state via hook
final_hidden = {}
def hook_final(module, input, output):
    h = output[0] if isinstance(output, tuple) else output
    final_hidden["h"] = h[0, -1, :].detach().clone()

handle = model_eager.model.layers[-1].register_forward_hook(hook_final)
with torch.no_grad():
    model_eager(**inputs)
handle.remove()

# Project through norm + unembedding
normed = model_eager.model.norm(
    final_hidden["h"].unsqueeze(0).unsqueeze(0)
).squeeze()
logits = model_eager.lm_head.weight @ normed

# Check all digit tokens
print("\nLogit values for digit strings at final layer (P3):")
print(f"{'Digit':>6}  {'Token IDs':>15}  {'Logit':>10}  {'Rank':>8}  Single?")
print("-" * 55)

all_logits_sorted = logits.argsort(descending=True)

for n in range(1, 21):
    ids    = tokenizer.encode(str(n), add_special_tokens=False)
    single = len(ids) == 1
    if single:
        logit_val = logits[ids[0]].item()
        rank      = (all_logits_sorted == ids[0]).nonzero().item() + 1
        print(f"{n:>6}  {str(ids):>15}  {logit_val:>10.4f}  {rank:>8}  YES")
    else:
        print(f"{n:>6}  {str(ids):>15}  {'N/A':>10}  {'N/A':>8}  NO — multi-token")

# Top-20 tokens by logit at final layer
print(f"\nTop-20 tokens at final layer (P3):")
top20 = logits.topk(20)
for i, (val, idx) in enumerate(zip(top20.values, top20.indices)):
    tok = tokenizer.decode([idx])
    print(f"  {i+1:>3}. {repr(tok):<20}  logit={val.item():.4f}")

PROBE DISSOCIATION DIAGNOSTIC — Qwen2.5-1.5B

Key question: is R2(repeated) high at the lock-in layers (L22-L25)?
Lock-in layers for Qwen 1.5B: L22 (MLP writes '8'), L24 (MLP writes '8')

 Layer    R2(rep)   R2(uniq)   MAE(rep)   MAE(uniq)    Dissociation?
----------------------------------------------------------------------
 embed    -0.2100    -0.2100     3.0000      3.0000                -
   L01     0.9023     0.7457     0.8342      1.1863                -
   L02     0.7336     0.5150     1.2315      1.5103                -
   L03     0.7253     0.7556     1.3739      1.1317                -
   L04     0.7069     0.8285     1.3194      0.8274                -
   L05     0.7687     0.8569     1.1719      0.7768                -
   L06     0.7239     0.8769     1.3072      0.7244                -
   L07     0.6874     0.8632     1.3564      0.7938                -
   L08     0.7076     0.8470     1.3389      0.8448                -
   L09     0.8044     0.8715     1.1035      0.7284

In [32]:
# Save updated Qwen 1.5B summary with diagnostic results
summary_qwen15_updated = {
    "model"                  : "Qwen/Qwen2.5-1.5B-Instruct",
    "probe_dissociation"     : {
        "verdict"            : "holds",
        "mean_r2_at_lockin"  : 0.9774,
        "min_r2_at_lockin"   : 0.9708,
        "lockin_layers"      : "L20-L26",
        "note"               : "weaker than Llama 1B (0.994) but above 0.95 threshold"
    },
    "logit_lens_limitation"  : {
        "issue"              : "all two-digit numbers are multi-token in Qwen tokenizer",
        "single_token_digits": [1, 2, 3, 4, 5, 6, 7, 8, 9],
        "implication"        : (
            "logit lens can track wrong-answer attractors (8, 6) "
            "but cannot track correct answer (10) or Llama 3B-style attractor (14). "
            "P3 outputting '10' correctly is autoregressive two-token generation, "
            "not visible in single-token logit lens."
        ),
        "verdict"            : "methodological artifact, not model failure"
    },
    "p3_final_layer_logits"  : {
        "top_single_token"   : "1 (rank 1, logit 33.5)",
        "explanation"        : (
            "'1' is top because it is the first token of '10'. "
            "Model correctly generates '1' then '0' autoregressively."
        )
    },
    "cross_family_notes"     : [
        "Probe dissociation holds — count encoded but not used at lock-in layers",
        "Logit lens limited to digits 1-9 due to Qwen tokenizer",
        "Wrong-answer attractors (8, 6) are single-token and fully trackable",
        "MLP writer at L22 confirmed at same relative depth as Llama 3B"
    ]
}

with open("summary_qwen2.5_1.5b_updated.json", "w") as f:
    json.dump(summary_qwen15_updated, f, indent=2)
print("Saved: summary_qwen2.5_1.5b_updated.json")
print("\nQwen 1.5B diagnostics complete.")
print("Both weaknesses resolved:")
print("  1. Probe dissociation holds (R2=0.977 at lock-in layers)")
print("  2. Logit lens gap is tokenizer artifact — '10' is multi-token in Qwen")

Saved: summary_qwen2.5_1.5b_updated.json

Qwen 1.5B diagnostics complete.
Both weaknesses resolved:
  1. Probe dissociation holds (R2=0.977 at lock-in layers)
  2. Logit lens gap is tokenizer artifact — '10' is multi-token in Qwen
